# Embeddings in LangChain

- **Package:** `langchain-huggingface` (`uv add langchain-huggingface`)
- **Under the hood:** Wraps `sentence-transformers` for LangChain
- **Interface:** `embed_documents(texts: list[str])` and `embed_query(text: str)`

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

texts = [
    "FastAPI is a Python web framework.",
    "FastAPI supports asynchronous programming."
]
# return list[list]
vectors = embeddings.embed_documents(texts) 
print(type(vectors)) 
print(len(vectors))      # num of vectors 
print(type(vectors[0]))
print(len(vectors[0]))   # embedding dimension (384)

query = "What is FastAPI?"
# return list[]
query_vector = embeddings.embed_query(query)

print(type(query_vector))
print(len(query_vector))


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7047.99it/s]


<class 'list'>
2
<class 'list'>
384
<class 'list'>
384


### 2. Model Inspection (`sentence_transformers`)
- `model.max_seq_length`: Maximum token capacity of the embedding model (256 tokens).
- `model.tokenizer`: Tokenizer used to verify chunk sizes against the model limits.

In [9]:
from sentence_transformers import SentenceTransformer

# 1. Load the underlying embedding model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Inspect maximum token capacity (Hard limit for chunk sizing)
print("=== 1. Model Constraints ===")
print("Max Token Capacity (max_seq_length):", model.max_seq_length)  # 256 tokens max per chunk

# 2. Sample text to evaluate
text = "FastAPI is a Python web framework."

# 3. Tokenize text using the model's tokenizer (BertTokenizer)
tokens = model.tokenizer.encode(text, add_special_tokens=False)

# 4. Compare Word Count vs Token Count
# Note: Token count is typically greater than word count due to subword splitting
word_count = len(text.split())
token_count = len(tokens)

print("\n=== 2. Tokenization Inspection ===")
print("Input Text :", text)
print("Token IDs  :", tokens)
print("Word Count :", word_count)
print("Token Count:", token_count, "(Always compare with chunk_size in length_function)")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 16955.00it/s]


=== 1. Model Constraints ===
Max Token Capacity (max_seq_length): 256

=== 2. Tokenization Inspection ===
Input Text : FastAPI is a Python web framework.
Token IDs  : [3435, 9331, 2072, 2003, 1037, 18750, 4773, 7705, 1012]
Word Count : 6
Token Count: 9 (Always compare with chunk_size in length_function)


### 3. Document Loading (`PyMuPDFLoader`)

In [10]:
from langchain_community.document_loaders import PyMuPDFLoader

# 1. Initialize loader and load all pages
loader = PyMuPDFLoader("Document_31.pdf")
documents = loader.load()

print("=== 1. Loaded Document Summary ===")
print(f"Total Pages Loaded: {len(documents)}\n")

# 2. Inspect raw page statistics
print("=== 2. Raw Page Statistics ===")
for doc in documents:
    page_num = doc.metadata.get("page")
    char_count = len(doc.page_content)
    print(f"Page {page_num:2d} | Characters: {char_count}")

# 3. Clean and sanitize metadata
# Keep only essential fields ('source', 'page') to prevent vector store payload bloat
for doc in documents:
    doc.metadata = {
        "source": doc.metadata.get("source"),
        "page": doc.metadata.get("page")
    }

print("\n=== 3. Sanitized Metadata Sample ===")
print("Page 0 Metadata:", documents[0].metadata)


=== 1. Loaded Document Summary ===
Total Pages Loaded: 5

=== 2. Raw Page Statistics ===
Page  0 | Characters: 1761
Page  1 | Characters: 1594
Page  2 | Characters: 1756
Page  3 | Characters: 1282
Page  4 | Characters: 1215

=== 3. Sanitized Metadata Sample ===
Page 0 Metadata: {'source': 'Document_31.pdf', 'page': 0}


### 4. Token-Aware Chunking
Using the model's tokenizer in `length_function` ensures every chunk strictly stays under `max_seq_length` (256 tokens).

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def token_length(text):
    # [CLS] [SEP] -> Classification, Separator
    return len(model.tokenizer.encode(text, add_special_tokens=False))

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,      # strictly under model limit (256 tokens)
    chunk_overlap=30,
    length_function=token_length
)

chunks = splitter.split_documents(documents) # return list[Document] has page_content , metadata
print(f"len of chunks {len(chunks)}")
for chunk in chunks[:10]:
    print(
        f"Chunk {chunk.metadata.get('page')} | "
        f"Tokens: {token_length(chunk.page_content)} | "
        f"Characters: {len(chunk.page_content)}"
    )


len of chunks 34
Chunk 0 | Tokens: 185 | Characters: 259
Chunk 0 | Tokens: 187 | Characters: 262
Chunk 0 | Tokens: 166 | Characters: 215
Chunk 0 | Tokens: 200 | Characters: 263
Chunk 0 | Tokens: 200 | Characters: 264
Chunk 0 | Tokens: 114 | Characters: 142
Chunk 0 | Tokens: 148 | Characters: 189
Chunk 0 | Tokens: 185 | Characters: 228
Chunk 1 | Tokens: 184 | Characters: 243
Chunk 1 | Tokens: 198 | Characters: 256


### 5. Embedding Document Chunks
> `embed_documents()` requires `list[str]`, not `list[Document]`. Extract `page_content` from chunks before embedding.

In [12]:
# Extract strings from Document objects
texts = [chunk.page_content for chunk in chunks]
vectors = embeddings.embed_documents(texts)  # needs list[str] not list[Document]

print(f"num of vector {len(vectors)}")
print(f"num of dimensions {len(vectors[0])}")


num of vector 34
num of dimensions 384


### 6. Asymmetric Embedding Models (Prefix Requirements)

Certain embedding models (e.g. `intfloat/e5-base-v2`, `BAAI/bge-small-en-v1.5`, `nomic-embed-text`) are trained asymmetrically and require explicit task prefixes:

| Type | Prefix Format | Example |
| :--- | :--- | :--- |
| **Query** | `query: ` | `"query: What is FastAPI?"` |
| **Passage / Document** | `passage: ` | `"passage: FastAPI is a modern Python framework..."` |

```text
all-MiniLM-L6-v2  ──▶ Symmetric  ──▶ No prefixes required
e5 / bge          ──▶ Asymmetric ──▶ Requires "query: " and "passage: " prefixes
```

## 7. Vector Store: Qdrant Integration (`langchain-qdrant`)

- **Package:** `langchain-qdrant` (`uv add langchain-qdrant`)
- **Prerequisite:** Running Qdrant server (`docker run -p 6333:6333 qdrant/qdrant`)

### How LangChain Maps `Document` to Qdrant `Point`

```text
LangChain Document                  Qdrant PointStruct
├── page_content   ──(embed)──▶     ├── vector (384-dim embedding)
└── metadata       ───────────▶     ├── payload (page_content + metadata)
                                    └── id (UUID auto-generated)
```

In [13]:
from langchain_qdrant import QdrantVectorStore  # interface for qdrant with langchain
from qdrant_client import QdrantClient

# 1. الاتصال المباشر لمسح الـ Collection المكررة
client = QdrantClient("http://localhost:6333")
client.delete_collection("langchain_demo")

ids = list(range(1, len(chunks) + 1))


vector_store = QdrantVectorStore.from_documents(
    documents=chunks,             # list[Document]
    ids=ids, 
    embedding=embeddings,          # embedding model
    url="http://localhost:6333",   # url of qdrant docker
    collection_name="langchain_demo",
)

### 7.2 Semantic Similarity Search

Querying the vector store to retrieve the top-$k$ most relevant chunks based on semantic similarity.

In [14]:
# Realistic query directly related to Document_31.pdf (EchoMind Cybersecurity & Medical Data Protection)
query = "ما هي خوارزمية تشفير الصور الطبية وملفات ECG؟"

# Top 2 most relevant chunks - Top K - K nearest results.
results = vector_store.similarity_search(query, k=2)  # returns list[Document]

print("Results count:", len(results))
print("Result type  :", type(results[0])) # type of first element inside list

for i, doc in enumerate(results):
    print(f"\n=== Result {i+1} (Page {doc.metadata.get('page')}) ===")
    print(doc.page_content.strip())
    print("\nMetadata:", doc.metadata)


Results count: 2
Result type  : <class 'langchain_core.documents.base.Document'>

=== Result 1 (Page 1) ===
صور ECG، تحليل الصور بالذكاء االصطناعي، إضافة مالحظات طبية. المطور: كل صالحيات المسؤول، والوصول إلى 
Django Admin. 
 
--- 
 
## 6.  حماية الملفات والصور الطبية 
 
 ****استراتيجية التشفير-

Metadata: {'source': 'Document_31.pdf', 'page': 1, '_id': 13, '_collection_name': 'langchain_demo'}

=== Result 2 (Page 0) ===
ECG  الطبية قبل تخزينها في قاعدة البيانات. عملية التشفير: قراءة بيانات الصورة الخام، تشفيرها بالمفتاح السري، تخزين
الناتج في BinaryField، وفك التشفير يت .طلب إدخال المفتاح الصحيح 
 
**PBKDF2 
 **لتجزئة كلمات المرور-

Metadata: {'source': 'Document_31.pdf', 'page': 0, '_id': 3, '_collection_name': 'langchain_demo'}


In [15]:
# Similarity search with distance scores
results_with_scores = vector_store.similarity_search_with_score(query, k=2) # list[tuple(doc,score)]

for doc, score in results_with_scores:
    print(f"Score: {score:.4f} | Page: {doc.metadata.get('page')}")
    print(doc.page_content[:150].strip(), "...")
    print("-" * 50)


Score: 0.7850 | Page: 1
صور ECG، تحليل الصور بالذكاء االصطناعي، إضافة مالحظات طبية. المطور: كل صالحيات المسؤول، والوصول إلى 
Django Admin. 
 
--- 
 
## 6.  حماية الملفات والص ...
--------------------------------------------------
Score: 0.7124 | Page: 0
ECG  الطبية قبل تخزينها في قاعدة البيانات. عملية التشفير: قراءة بيانات الصورة الخام، تشفيرها بالمفتاح السري، تخزين
الناتج في BinaryField، وفك التشفير ...
--------------------------------------------------


### 7.3 Metadata Filtering (`qdrant_client.models.Filter`)

Filters restrict vector search to documents matching specific payload attributes (e.g., specific page, author, or source) *before* computing vector similarity.

In [16]:
from qdrant_client.models import Filter, FieldCondition, MatchValue # there is filterselector

# فلترة النتائج بحيث ترجع فقط من الصفحة رقم 1
filter = Filter(
    must=[
        FieldCondition(
            key="metadata.page",
            match=MatchValue(value=1)
        )
    ]
)

query = "ما هي خوارزمية تشفير الصور الطبية وملفات ECG؟"

results = vector_store.similarity_search_with_score(
    query,
    k=2,
    filter=filter
)

for doc, score in results:
    print(f"Score: {score:.4f} | Page: {doc.metadata.get('page')}")
    print(doc.page_content[:150].strip(), "...")
    print("-" * 50)


Score: 0.7850 | Page: 1
صور ECG، تحليل الصور بالذكاء االصطناعي، إضافة مالحظات طبية. المطور: كل صالحيات المسؤول، والوصول إلى 
Django Admin. 
 
--- 
 
## 6.  حماية الملفات والص ...
--------------------------------------------------
Score: 0.5870 | Page: 1
.مع رسالة خطأ مخصصة 
 
 ****نظام صالحيات المستخدمين- 
 المستخدم العادي: تسجيل دخول، رفع صور ECG،  عرض تاريخه الطبي، التواصل مع
 الدعم. المسؤول: كل صال ...
--------------------------------------------------


## 8. The Retriever Abstraction (`as_retriever`)

A **Retriever** wraps the vector store in a standardized **`Runnable`** interface.

```text
Query (str)  ──▶  [ Retriever (Runnable) ]  ──▶  list[Document]
```

### Key Architectural Advantages:
1. **Uniform Interface:** Accepts a raw string query (`str`) and internally handles embedding and querying.
2. **LCEL Compatibility:** Directly connects via pipe `|` into chains: `retriever | format_docs | prompt | llm`.
3. **Multiple Search Strategies:** Configurable via `search_type` parameter.

| `search_type` | Strategy | Primary Use Case |
| :--- | :--- | :--- |
| **`similarity`** (Default) | Standard nearest neighbors by cosine similarity | Retrieve highest scoring matches regardless of overlap. |
| **`mmr`** | Maximal Marginal Relevance (Relevance + Diversity) | Avoid redundant chunks covering identical information. |
| **`similarity_score_threshold`** | Minimum confidence score cutoff | Prevent hallucination by returning only high-confidence chunks. |

In [17]:
# 1. Standard Similarity Retriever
retriever_sim = vector_store.as_retriever(
    search_type="similarity",      # default strategy
    search_kwargs={"k": 3}         # top-3 most similar chunks
)

docs = retriever_sim.invoke("ما هي خوارزمية تشفير الصور الطبية وملفات ECG؟")

print("Type :", type(docs))
print("Count:", len(docs))
for i, doc in enumerate(docs):
    print(f"Result {i} | {doc.page_content[:40].strip()}... | Page {doc.metadata.get('page')}")


Type : <class 'list'>
Count: 3
Result 0 | صور ECG، تحليل الصور بالذكاء االصطناعي،... | Page 1
Result 1 | ECG  الطبية قبل تخزينها في قاعدة البيانا... | Page 0
Result 2 | .)الملف الشخصي، حذف الحساب 
 
--- 
 
 ##... | Page 3


### 8.2 Maximal Marginal Relevance (MMR)

MMR selects chunks that are both **relevant** to the query and **diverse** from each other, eliminating repetitive results.

```text
                      lambda_mult Parameter
0.0 (Max Diversity) <──────────────────────────> 1.0 (Pure Relevance)
```

- **`fetch_k`**: Size of the initial candidate pool evaluated for similarity (e.g. 10).
- **`k`**: Final number of diverse chunks returned (e.g. 3).
- **`lambda_mult`**: Balance between diversity (`0.0`) and similarity (`1.0`). Defaults to `0.5`.

In [18]:
# 2. MMR Retriever (Balancing Relevance & Diversity)
retriever_mmr = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,            # return top-3 diverse chunks
        "fetch_k": 10,     # fetch 10 candidates first
        "lambda_mult": 0.5 # 0 -> maximum diversity | 1 -> pure relevance
    }
)

docs_mmr = retriever_mmr.invoke("ما هي خوارزمية تشفير الصور الطبية وملفات ECG؟")

for i, doc in enumerate(docs_mmr):
    print(f"Result {i} | {doc.page_content[:40].strip()}... | Page {doc.metadata.get('page')}")


Result 0 | صور ECG، تحليل الصور بالذكاء االصطناعي،... | Page 1
Result 1 | --- 
 
 ##
1
 . خوارزميات التشفير المستخ... | Page 0
Result 2 | ECG  الطبية قبل تخزينها في قاعدة البيانا... | Page 0


### 8.3 Score Threshold Filtering (`similarity_score_threshold`)

Discards any chunks with similarity scores below `score_threshold`. If no documents meet the threshold, returns an empty list—protecting the LLM from generating answers on irrelevant data.

In [19]:
# 3. Similarity Score Threshold Retriever
retriever_threshold = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 3,
        "score_threshold": 0.70  # Only return chunks with similarity >= 0.70
    }
)

docs_threshold = retriever_threshold.invoke("ما هي خوارزمية تشفير الصور الطبية وملفات ECG؟")

print("Chunks passing threshold:", len(docs_threshold))
for i, doc in enumerate(docs_threshold):
    print(f"Result {i} | {doc.page_content[:40].strip()}... | Page {doc.metadata.get('page')}")


Chunks passing threshold: 3
Result 0 | صور ECG، تحليل الصور بالذكاء االصطناعي،... | Page 1
Result 1 | ECG  الطبية قبل تخزينها في قاعدة البيانا... | Page 0
Result 2 | .)الملف الشخصي، حذف الحساب 
 
--- 
 
 ##... | Page 3


## 9. Building the End-to-End RAG Chain (LCEL)

Connecting the **Retriever** with a **ChatPromptTemplate** and **ChatModel** using the pipe (`|`) operator.

In [20]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader('article.txt')

doc = loader.load() #doc = [ Document(page_content="...") ]

print(type(doc))
print(len(doc))
print(f"you page_content: {doc[0].page_content[0:10]} |  your metadata: {doc[0].metadata}")

<class 'list'>
1
you page_content: Welcome to |  your metadata: {'source': 'article.txt'}


In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def token_length(text):
        return len(model.tokenizer.encode(text, add_special_tokens=False))
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,      # strictly under model limit (256 tokens)
    chunk_overlap=30,
    length_function=token_length
)        
chunks = splitter.split_documents(doc) # list[doc]

print(len(chunks))            
print(type(chunks[0]))          
print(chunks[0].metadata) 

6
<class 'langchain_core.documents.base.Document'>
{'source': 'article.txt'}


In [22]:
from langchain_qdrant import QdrantVectorStore

start_id = 35

ids = list(range(
    start_id, # 35
    start_id + len(chunks) #41
))


vector_store.add_documents(
    documents=chunks,
    ids=ids
)

[35, 36, 37, 38, 39, 40]

In [23]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)
query = "Can I use large language models to write Wikipedia articles?"

docs = retriever.invoke(query) #list[docs]

for i, doc in enumerate(docs):
    print(f"\n=== Result {i} | id: {doc.metadata['_id'] }| Source: {doc.metadata.get('source')} ===")
    print(doc.page_content)
    print("-" * 50)



=== Result 0 | id: 35| Source: article.txt ===
Welcome to Wikipedia! Creating a new article is one of many ways to contribute to the encyclopedia, but can be difficult for new editors. We recommend looking at our introductory tutorials or contributing to Wikipedia to learn the basics about editing first. Working on existing articles is a great way to learn Wikipedia's protocols and style conventions; see the Task Center or your homepage for articles that need your assistance and tasks you can help out with.

Before creating your first article, it's highly recommended to practice in your sandbox. Choose a topic supported by reliable, independent sources. Write in a neutral tone, avoid promotional language, and make sure all the information in the article is supported by your sources. Don't use large language models to write your articles either. If you're unsure, the Article Wizard can help guide you step-by-step through the process. Don't worry if it's not perfect—every experienced ed

In [24]:
context = "\n\n".join(
    doc.page_content
     for doc in docs
)
print(context)

Welcome to Wikipedia! Creating a new article is one of many ways to contribute to the encyclopedia, but can be difficult for new editors. We recommend looking at our introductory tutorials or contributing to Wikipedia to learn the basics about editing first. Working on existing articles is a great way to learn Wikipedia's protocols and style conventions; see the Task Center or your homepage for articles that need your assistance and tasks you can help out with.

Before creating your first article, it's highly recommended to practice in your sandbox. Choose a topic supported by reliable, independent sources. Write in a neutral tone, avoid promotional language, and make sure all the information in the article is supported by your sources. Don't use large language models to write your articles either. If you're unsure, the Article Wizard can help guide you step-by-step through the process. Don't worry if it's not perfect—every experienced editor started with their first try!

Once you are

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """Answer the question using only the provided context.

Context:
{context}"""
    ),
    (
        "human",
        "{question}"
    )
])
# context and query in the upper cells
formatted_prompt = prompt.invoke({
    "context": context,
    "question": query
})
print(formatted_prompt)

messages=[SystemMessage(content="Answer the question using only the provided context.\n\nContext:\nWelcome to Wikipedia! Creating a new article is one of many ways to contribute to the encyclopedia, but can be difficult for new editors. We recommend looking at our introductory tutorials or contributing to Wikipedia to learn the basics about editing first. Working on existing articles is a great way to learn Wikipedia's protocols and style conventions; see the Task Center or your homepage for articles that need your assistance and tasks you can help out with.\n\nBefore creating your first article, it's highly recommended to practice in your sandbox. Choose a topic supported by reliable, independent sources. Write in a neutral tone, avoid promotional language, and make sure all the information in the article is supported by your sources. Don't use large language models to write your articles either. If you're unsure, the Article Wizard can help guide you step-by-step through the process.

In [26]:
from langchain_anthropic import ChatAnthropic
from dotenv import load_dotenv, find_dotenv
import os 

load_dotenv(find_dotenv())
anthropic_key = os.getenv("ANTHROPIC_API_KEY")

# Initialize ChatModel Component
llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    temperature=0,
    max_tokens=500,
    api_key=anthropic_key
)

# Unified Interface invocation
response = llm.invoke(formatted_prompt)
print(response.content)

No, according to the provided context, you should not use large language models to write Wikipedia articles. The context explicitly states: "Don't use large language models to write your articles either."


### 9.2 The Input Branching Pattern (`RunnableParallel` & `RunnablePassthrough`)

In RAG, the user provides a **single input** (the question string), but `ChatPromptTemplate` requires **two variables**: `{context}` and `{question}`.

`RunnableParallel` splits (forks) the incoming query into two parallel branches and merges them into a dictionary for the prompt:

```text
                               ┌──▶ retriever ──▶ format_docs ──▶ {context}  ┐
query ("Can I use LLMs...?") ──┤                                             ├──▶ Prompt ──▶ LLM ──▶ Answer
                               └──▶ RunnablePassthrough()     ──▶ {question} ┘
```

| Branch | Runnable Chain | Output |
| :--- | :--- | :--- |
| **`context`** | `retriever | format_docs` | Relevant chunks retrieved from Qdrant and formatted as a single string. |
| **`question`** | `RunnablePassthrough()` | The original user query passed through unchanged. |


In [27]:
from langchain_core.runnables import RunnableLambda,RunnableParallel,RunnablePassthrough
def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )
format_docs_runnable = RunnableLambda(format_docs)

rag_input =RunnableParallel(
    context=retriever | format_docs_runnable,      # return list[doc] then return context only
    question=RunnablePassthrough(),
    documents=retriever # return list[doc] 
)
result = rag_input.invoke("Can I use large language models to write Wikipedia articles?")

print(type(result)) 
print(result.keys()) 
print("Context:")

print(" ==========your context return=========")
print(result["context"])
print(" ==========your question return=========")
print(result["question"])
print(" ==========your documents return=========")
print(result["documents"])


<class 'dict'>
dict_keys(['context', 'question', 'documents'])
Context:
 ==========your context return=========
Welcome to Wikipedia! Creating a new article is one of many ways to contribute to the encyclopedia, but can be difficult for new editors. We recommend looking at our introductory tutorials or contributing to Wikipedia to learn the basics about editing first. Working on existing articles is a great way to learn Wikipedia's protocols and style conventions; see the Task Center or your homepage for articles that need your assistance and tasks you can help out with.

Before creating your first article, it's highly recommended to practice in your sandbox. Choose a topic supported by reliable, independent sources. Write in a neutral tone, avoid promotional language, and make sure all the information in the article is supported by your sources. Don't use large language models to write your articles either. If you're unsure, the Article Wizard can help guide you step-by-step through t

In [28]:
sources = [
    doc.metadata
    for doc in result["documents"]
]
print(sources)


[{'source': 'article.txt', '_id': 35, '_collection_name': 'langchain_demo'}, {'source': 'article.txt', '_id': 36, '_collection_name': 'langchain_demo'}, {'source': 'article.txt', '_id': 40, '_collection_name': 'langchain_demo'}]


In [ ]:
#           cell[27], cell[25], cell[26]
rag_chain = rag_input| prompt | llm # invoke each ather step by step 
response = rag_chain.invoke(
   "Can I use large language models to write Wikipedia articles?"
)
print(response)
print(response.content)


content='No, according to the provided context, you should not use large language models to write Wikipedia articles. The context explicitly states: "Don\'t use large language models to write your articles either."' additional_kwargs={} response_metadata={'id': 'msg_011CfpzEg3oxHtPaN16fEK2b', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 325, 'output_tokens': 41, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'diagnostics': None, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'} id='lc_run--01a11c13-ff21-7682-a75c-22f862a0267c-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 325, 'output_tokens': 41, 'total_tokens': 3

In [30]:
final_result = {
    "answer": response.content,
    "sources": sources
}

print(final_result)


{'answer': 'No, according to the provided context, you should not use large language models to write Wikipedia articles. The context explicitly states: "Don\'t use large language models to write your articles either."', 'sources': [{'source': 'article.txt', '_id': 35, '_collection_name': 'langchain_demo'}, {'source': 'article.txt', '_id': 36, '_collection_name': 'langchain_demo'}, {'source': 'article.txt', '_id': 40, '_collection_name': 'langchain_demo'}]}


### 10. Unified End-to-End RAG Pipeline (Answer + Citations)

This pipeline automates the entire RAG workflow into a single LCEL chain that returns both the **generated answer** and its **source citations**:

```text
                                  ┌──▶ answer : extract {context, question} ──▶ prompt ──▶ llm ──▶ .content
query ──▶ rag_input (dict) ──▶ RunnableParallel ─┤
                                  └──▶ sources: extract metadata from [documents]
```

#### How it works:
1. **Input Branching (`rag_input`):** Takes the query string, queries Qdrant for relevant chunks, and prepares a dictionary containing `context`, `question`, and `documents`.
2. **Parallel Output (`RunnableParallel`):**
   - **`answer` branch:** Extracts `context` and `question`, pipes them through `prompt | llm`, and extracts the text response (`.content`).
   - **`sources` branch:** Extracts metadata (`doc.metadata`) from each retrieved chunk to provide verifiable source citations.
3. **Unified Output:** A single `.invoke()` produces `{"answer": "...", "sources": [...]}`.


In [ ]:
from langchain_core.runnables import RunnableParallel, RunnableLambda

rag_chain = (
    rag_input
    | RunnableParallel(
        answer=(
            RunnableLambda(
                lambda x: { 
                    "context": x["context"],
                    "question": x["question"]
                }
            )
            | prompt
            | llm
            | RunnableLambda(lambda x: x.content)
        ),
        sources=RunnableLambda(
            lambda x: [
                doc.metadata
                for doc in x["documents"]
            ]
        )
    )
)
result = rag_chain.invoke(
    "Can I use large language models to write Wikipedia articles?"
)

print(result)

{'answer': 'No, according to the provided context, you should not use large language models to write Wikipedia articles. The context explicitly states: "Don\'t use large language models to write your articles either."', 'sources': [{'source': 'article.txt', '_id': 35, '_collection_name': 'langchain_demo'}, {'source': 'article.txt', '_id': 36, '_collection_name': 'langchain_demo'}, {'source': 'article.txt', '_id': 40, '_collection_name': 'langchain_demo'}]}


### 11. Streaming Pipeline Output (`.stream()`)
Yields dictionary chunks as the parallel branches (`sources`, `answer`) complete.


In [32]:
# Stream pipeline output chunks
query = "Can I use large language models to write Wikipedia articles?"

for chunk in rag_chain.stream(query):
    print(chunk)


{'sources': [{'source': 'article.txt', '_id': 35, '_collection_name': 'langchain_demo'}, {'source': 'article.txt', '_id': 36, '_collection_name': 'langchain_demo'}, {'source': 'article.txt', '_id': 40, '_collection_name': 'langchain_demo'}]}
{'answer': 'No, according to the provided context, you should not use large language models to write Wikipedia articles. The context explicitly states: "Don\'t use large language models to write your articles either."'}


### Real-Time Token Streaming (`.astream()`)
Streams text tokens directly from the LLM as they are generated (`flush=True` prevents output buffering).


In [ ]:
# Stream text tokens directly from the LLM
# Note: prompt auto-extracts {context, question} and silently ignores extra keys like 'documents'
stream_chain = rag_input | prompt | llm

query = "Can I use large language models to write Wikipedia articles?"

async for chunk in stream_chain.astream(query):
    print(chunk.content, end="", flush=True)


I cannot answer this question based on the provided context. The context only contains information about security procedures, OTP types, password hashing (PBKDF2), and encryption methods used in what appears to be a Django-based system. It does not mention FastAPI.

To answer your question about FastAPI, I would need relevant context about that framework.

### 12. Structured Outputs with Pydantic

By default, LLMs return unstructured strings. For production APIs and databases, applications require **strictly typed data structures**.

LangChain's `.with_structured_output(Schema)` enforces the model to return a validated Pydantic object (using Tool Calling / JSON Schema under the hood):

- **Input:** Prompt / Query
- **Contract:** Defined via Pydantic `BaseModel` (`answer: str`, `confidence: float`)
- **Output:** Validated Python object with dot-accessible attributes (`result.answer`, `result.confidence`)


In [37]:
from pydantic import BaseModel


class Answer(BaseModel):
    answer: str
    confidence: float
# return structured output 
structured_llm = llm.with_structured_output(Answer) # schema -> LLM -> output -> validaton by pydantic

result = structured_llm.invoke(
    "Explain FastAPI briefly and give your confidence."
)

In [35]:
print(result.answer)
print(result.confidence)

FastAPI is a modern, fast (high-performance) web framework for building APIs with Python 3.6+. Here are its key characteristics:

**Core Features:**
- **Speed**: Built on Starlette (ASGI framework) and Pydantic, it's one of the fastest Python web frameworks, comparable to Node.js and Go
- **Easy to Use**: Simple and intuitive syntax that reduces development time and bugs
- **Automatic Documentation**: Automatically generates interactive API documentation (Swagger UI and ReDoc) from your code
- **Type Hints**: Leverages Python type hints for data validation, serialization, and documentation
- **Data Validation**: Uses Pydantic models for automatic request/response validation
- **Async Support**: Built-in support for async/await, enabling high concurrency
- **Security**: Includes built-in security features like OAuth2, API keys, and CORS
- **Production Ready**: Can be deployed with ASGI servers like Uvicorn, Gunicorn, or Hypercorn

**Common Use Cases:**
- RESTful APIs
- Microservices
- R

### 12.2 Production-Ready RAG: Structured Output + Source Citations

Here we integrate **Pydantic Validation** into the full **LCEL RAG Pipeline**.

The pipeline executes two parallel tasks in a single invocation:
1. **`answer` branch:** Passes retrieved context and question to `structured_llm` to enforce the `RAGAnswer` schema.
2. **`sources` branch:** Extracts document metadata (`page`, `source`, `_id`) for auditable citations.


In [36]:
from pydantic import BaseModel

class RAGAnswer(BaseModel):
    answer: str

structured_llm = llm.with_structured_output(RAGAnswer)

answer_chain = (
    RunnableLambda(
        lambda x: {
            "context": x["context"],
            "question": x["question"]
        }
    )
    | prompt
    | structured_llm
)

rag_chain = (
    rag_input
    | RunnableParallel(
        answer=answer_chain,
        sources=RunnableLambda(
            lambda x: [
                doc.metadata
                for doc in x["documents"]
            ]
        )
    )
)

result = rag_chain.invoke(
    "Can I use large language models to write Wikipedia articles?"
)

print(result)

{'answer': RAGAnswer(answer='No, you should not use large language models to write Wikipedia articles. According to Wikipedia\'s guidelines for creating new articles, contributors are explicitly advised: "Don\'t use large language models to write your articles either."'), 'sources': [{'source': 'article.txt', '_id': 35, '_collection_name': 'langchain_demo'}, {'source': 'article.txt', '_id': 36, '_collection_name': 'langchain_demo'}, {'source': 'article.txt', '_id': 40, '_collection_name': 'langchain_demo'}]}
